In [2]:
!pip install accelerate  # Optional: for fast GPU usage

  Cloning https://github.com/allenai/PRIMERA.git to c:\users\adidya\appdata\local\temp\pip-req-build-vxy4kyf7


  Running command git clone --filter=blob:none --quiet https://github.com/allenai/PRIMERA.git 'C:\Users\Adidya\AppData\Local\Temp\pip-req-build-vxy4kyf7'
  remote: Repository not found.
  fatal: repository 'https://github.com/allenai/PRIMERA.git/' not found
  error: subprocess-exited-with-error
  
  git clone --filter=blob:none --quiet https://github.com/allenai/PRIMERA.git 'C:\Users\Adidya\AppData\Local\Temp\pip-req-build-vxy4kyf7' did not run successfully.
  exit code: 128
  
  See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

git clone --filter=blob:none --quiet https://github.com/allenai/PRIMERA.git 'C:\Users\Adidya\AppData\Local\Temp\pip-req-build-vxy4kyf7' did not run successfully.
exit code: 128

See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Invalid requirement: '#': Expected package name at the start of dependenc

In [19]:
import datasets
print(datasets.__version__)
# Should print something like 2x.x.x (e.g., 2.19.0, 2.18.0, etc.)

2.19.0


In [25]:
from datasets import load_dataset

dataset = load_dataset("multi_news", trust_remote_code=True, cache_dir="internship model code")
train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

from transformers import AutoTokenizer

model_name = "allenai/PRIMERA"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 4096  # PRIMERA supports long inputs
max_target_length = 256

def preprocess_function(batch):
    inputs = tokenizer(batch["document"], max_length=max_input_length, truncation=True, padding="max_length")
    targets = tokenizer(batch["summary"], max_length=max_target_length, truncation=True, padding="max_length")
    inputs["labels"] = targets["input_ids"]
    return inputs

train_dataset = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
val_dataset = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)
test_dataset = test_data.map(preprocess_function, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/44972 [00:00<?, ? examples/s]

Map:   0%|          | 0/5622 [00:00<?, ? examples/s]

Map:   0%|          | 0/5622 [00:00<?, ? examples/s]

In [26]:
print("Train set size:", len(train_data))
print("Validation set size:", len(val_data))
print("Test set size:", len(test_data))

Train set size: 44972
Validation set size: 5622
Test set size: 5622


In [27]:
# Select only 1,000 samples for training and 100 each for validation and test
train_data = train_data.select(range(1000))
val_data = val_data.select(range(100))
test_data = test_data.select(range(100))

# Re-run preprocessing for the sampled data
train_dataset = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
val_dataset = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)
test_dataset = test_data.map(preprocess_function, batched=True, remove_columns=test_data.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [28]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

batch_size = 1  # Start with 1 for PRIMERA, increase if GPU allows

training_args = Seq2SeqTrainingArguments(
    output_dir="./primera_finetuned",
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=5000,
    eval_steps=5000,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=3e-5,
    num_train_epochs=1,           # Set to 2-3 for best results if time/VRAM allows
    predict_with_generate=True,
    logging_steps=1000,
    save_total_limit=2,
    fp16=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

C:\Users\Adidya\anaconda3\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [33]:
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=1000, training_loss=2.066462890625, metrics={'train_runtime': 9243.1906, 'train_samples_per_second': 0.108, 'train_steps_per_second': 0.108, 'total_flos': 9597037314048000.0, 'train_loss': 2.066462890625, 'epoch': 1.0})

In [35]:
results = trainer.evaluate(val_dataset)
print(results)

{'eval_loss': 1.8384735584259033, 'eval_runtime': 175.2551, 'eval_samples_per_second': 0.571, 'eval_steps_per_second': 0.571, 'epoch': 1.0}


In [36]:
from tqdm import tqdm

n_samples = 10  # or any number you like (keep it small to start)
generated_summaries = []
reference_summaries = []

for example in tqdm(val_data.select(range(n_samples))):
    input_ids = tokenizer(
        example["document"],
        max_length=max_input_length,
        truncation=True,
        return_tensors="pt"
    ).input_ids.to("cuda")

    output_ids = model.generate(
        input_ids=input_ids,
        max_length=max_target_length,
        num_beams=4
    )

    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_summaries.append(summary)
    reference_summaries.append(example["summary"])

# Evaluate ROUGE
from datasets import load_metric
rouge = load_metric("rouge")

scores = rouge.compute(predictions=generated_summaries, references=reference_summaries)
print(scores)

100%|██████████| 10/10 [09:13<00:00, 55.30s/it]
C:\Users\Adidya\AppData\Local\Temp\ipykernel_11688\4132971940.py:27: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  rouge = load_metric("rouge")
C:\Users\Adidya\anaconda3\Lib\site-packages\datasets\load.py:759: FutureWarning: The repository for rouge contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.19.0/metrics/rouge/rouge.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


{'rouge1': AggregateScore(low=Score(precision=0.44642153070833934, recall=0.3601315776720804, fmeasure=0.4047408259725828), mid=Score(precision=0.4944320329490416, recall=0.4060552326941627, fmeasure=0.44050944983277907), high=Score(precision=0.5412740097076582, recall=0.4537208857925725, fmeasure=0.46856308655437606)), 'rouge2': AggregateScore(low=Score(precision=0.11545996061826307, recall=0.09318138021129413, fmeasure=0.10410177660465597), mid=Score(precision=0.14418762292638515, recall=0.11996976347685348, fmeasure=0.12933293266109597), high=Score(precision=0.17249388070514338, recall=0.1494429962460941, fmeasure=0.15449931310870232)), 'rougeL': AggregateScore(low=Score(precision=0.19356694240738848, recall=0.16022014713953342, fmeasure=0.17657054454401822), mid=Score(precision=0.22378692112787793, recall=0.18271239056144878, fmeasure=0.19865121600137217), high=Score(precision=0.27349948090735465, recall=0.2061053563022637, fmeasure=0.2219181955401175)), 'rougeLsum': AggregateScore

In [38]:
print(f"ROUGE-1: {scores['rouge1'].mid.fmeasure:.4f}")
print(f"ROUGE-2: {scores['rouge2'].mid.fmeasure:.4f}")
print(f"ROUGE-L: {scores['rougeL'].mid.fmeasure:.4f}")

ROUGE-1: 0.4405
ROUGE-2: 0.1293
ROUGE-L: 0.1987


In [37]:
trainer.save_model("./primera_finetuned")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'no_repeat_ngram_size': 3}
